In [1]:
# 경찰청 범죄 발생 지역별 통계 csv파일 정제

from pathlib import Path
import pandas as pd

raw_path = Path("경찰청_범죄 발생 지역별 통계_20251231.csv")
out_path = Path("crime_stats_clean.csv")

# 1. CP949 원본 읽기
df = pd.read_csv(raw_path, encoding="cp949")

# 앞뒤 공백 정리 (칼럼명 및 문자열 값)
df.columns = [col.strip() for col in df.columns]
df["범죄대분류"] = df["범죄대분류"].astype(str).str.strip()
df["범죄중분류"] = df["범죄중분류"].astype(str).str.strip()

# 2. Wide -> Long 변환 (38개 범죄중분류 x 246개 지역 = 9,348행)
melted = df.melt(
    id_vars=["범죄대분류", "범죄중분류"],
    var_name="지역전체",
    value_name="발생건수",
)
melted["지역전체"] = melted["지역전체"].astype(str).str.strip()

# 3. 지역명 분리 ('서울 종로구' -> '서울', '종로구' / '세종시' -> '세종시', '')
split_region = melted["지역전체"].str.split(" ", n=1, expand=True)
melted["시도_구분"] = split_region[0].str.strip()
melted["시군구_상세"] = split_region[1].fillna("").str.strip()

# 4. 데이터 타입 및 이상치·결측치·중복 점검
melted["발생건수"] = pd.to_numeric(melted["발생건수"], errors="coerce")

print("변환 후 행/열:", melted.shape)
print("결측치 수:\n", melted.isna().sum())
print("중복 행 수:", melted.duplicated().sum())
print("음수 발생건수 수:", (melted["발생건수"] < 0).sum())
print("발생건수 기초통계:\n", melted["발생건수"].describe())

# 정수형 변환 및 이상치(음수) 제거
melted = melted[melted["발생건수"] >= 0].copy()
melted["발생건수"] = melted["발생건수"].astype("int64")

# 5. MariaDB 적재용 UTF-8(BOM) CSV 저장
melted.to_csv(out_path, index=False, encoding="utf-8-sig")
print("저장 완료:", out_path.resolve())

변환 후 행/열: (9348, 6)
결측치 수:
 범죄대분류     0
범죄중분류     0
지역전체      0
발생건수      0
시도_구분     0
시군구_상세    0
dtype: int64
중복 행 수: 0
음수 발생건수 수: 0
발생건수 기초통계:
 count     9348.000000
mean       171.126979
std        581.943269
min          0.000000
25%          1.000000
50%          7.000000
75%         62.000000
max      10984.000000
Name: 발생건수, dtype: float64
저장 완료: C:\Users\asas\Desktop\공부\AI 데이터 분석\python\crime_stats_clean.csv
